In [10]:
import torch
print(torch.cuda.is_available())

device = 'cuda' if torch.cuda.is_available() else 'cpu'

True


Optimizing Model Parameters

In [11]:
# Prerequisite Code

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# transforms v1) .ToTensor(): normalize and convert to tensor
# transforms v2) .ToImage(): convert to Image Tensor
#                .ToDtype(): convert to specific dtype and scale the values 
                

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),  # Layer 1
            nn.ReLU(),              # ReLU Activation
            nn.Linear(512, 512),    # Layer 2
            nn.ReLU(),              # ReLU Activation
            nn.Linear(512, 10),     # Output Layer
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)

In [3]:
# Hyperparameters

# Number of epochs: the number of times to iterate over the dataset
# Batch size: the number of data samples propagated through the network 
#             before the parameters are updated
# Learning rate: how much to update models parameters at each batch/epoch.
#                Smaller values yields slow learning, 
#                while larger values may result in unpredictable behavior

learning_rate = 1e-3
batch_size = 64
epochs = 5

In [4]:
# Optimization Loop

# Each iteration of the optimization loop is called an epoch.
# Each epoch consists of two main parts
#       1. Training: the model learns from the training dataset
#       2. Testing: the model is evaluated on the test dataset
print("Optimization Loop")

Optimization Loop


In [5]:
# Loss function

# Loss function measures the degree of dissimilarity 
# of obtained result to the target value

# the loss function that we want to minimize during training
# Common loss functions
#       nn.MSELoss: Mean Squared Error Loss for regression tasks 
#       nn.NLLLoss: Negative Log Likelihood Loss for classification tasks
#       nn.CrossEntropyLoss: combines nn.LogSoftmax() and nn.NLLLoss()


# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

In [6]:
# Optimizer

# Optimization algorithms define how this process is performed 
# All optimization logic is encapsulated in the optimizer object

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Inside the training loop, optimization happens in three steps:

- Call `optimizer.zero_grad()` to reset the gradients of model parameters. Gradients by default add up; to prevent double-counting, we explicitly zero them at each iteration.
- Backpropagate the prediction loss with a call to `loss.backward()`. PyTorch deposits the gradients of the loss w.r.t. each parameter.
- Once we have our gradients, we call `optimizer.step()` to adjust the parameters by the gradients collected in the backward pass.


In [13]:
# Full Implementation

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    # enumerate is python built-in function that include the index of the batch in the loop
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.307276 [   64/60000]
loss: 2.296965 [ 6464/60000]
loss: 2.272311 [12864/60000]
loss: 2.263205 [19264/60000]
loss: 2.251167 [25664/60000]
loss: 2.215764 [32064/60000]
loss: 2.223081 [38464/60000]
loss: 2.189946 [44864/60000]
loss: 2.185652 [51264/60000]
loss: 2.143197 [57664/60000]
Test Error: 
 Accuracy: 43.2%, Avg loss: 2.146430 

Epoch 2
-------------------------------
loss: 2.160192 [   64/60000]
loss: 2.153430 [ 6464/60000]
loss: 2.088091 [12864/60000]
loss: 2.104350 [19264/60000]
loss: 2.049981 [25664/60000]
loss: 1.991109 [32064/60000]
loss: 2.012429 [38464/60000]
loss: 1.938206 [44864/60000]
loss: 1.944973 [51264/60000]
loss: 1.858921 [57664/60000]
Test Error: 
 Accuracy: 60.6%, Avg loss: 1.862825 

Epoch 3
-------------------------------
loss: 1.899996 [   64/60000]
loss: 1.871497 [ 6464/60000]
loss: 1.742993 [12864/60000]
loss: 1.783376 [19264/60000]
loss: 1.671488 [25664/60000]
loss: 1.632980 [32064/60000]
loss: 1.638688 [38464/